# Naive Bayes - Tiny Spam Classifier

Naive Bayes is a simple probabilistic classifier that applies Bayes' rule with the (naive) assumption that features (words) are conditionally independent given the class. We use a multinomial model over word counts to classify brief SMS-like messages as `spam` or `ham`.

In [ ]:
import numpy as np
from collections import Counter

In [ ]:
# Toy dataset: short messages and labels
texts = [
    'win a free prize now',
    'claim your free prize',
    'free tickets for you',
    'urgent call now',
    'meeting at 3pm',
    'project due tomorrow',
    'lets have lunch',
    'can we reschedule meeting',
    'see you at the office'
]
labels = ['spam','spam','spam','spam','ham','ham','ham','ham','ham']

In [ ]:
# Preprocessing: lowercase and whitespace split
def tokenize(s):
    return s.lower().split()

tokenized = [tokenize(t) for t in texts]
vocab = sorted({w for doc in tokenized for w in doc})
w2i = {w:i for i,w in enumerate(vocab)}

def to_bow(doc):
    c = np.zeros(len(vocab), dtype=int)
    for w in doc:
        c[w2i[w]] += 1
    return c

X = np.vstack([to_bow(d) for d in tokenized])
y = np.array(labels)
print('Vocab:', vocab)
print('X shape:', X.shape, 'y:', y)

In [ ]:
# Training: estimate class priors and P(word|class) with Laplace smoothing
classes, counts = np.unique(y, return_counts=True)
priors = {c: counts[i] / len(y) for i,c in enumerate(classes)}

alpha = 1.0  # Laplace smoothing
word_counts = {c: np.zeros(len(vocab), dtype=float) for c in classes}
total_words = {c: 0.0 for c in classes}
for xi, label in zip(X, y):
    word_counts[label] += xi
    total_words[label] += xi.sum()

# likelihoods P(w|y) with smoothing, stored in log form
log_pw_given = {}
for c in classes:
    num = word_counts[c] + alpha
    den = total_words[c] + alpha * len(vocab)
    log_pw_given[c] = np.log(num / den)

log_priors = {c: np.log(priors[c]) for c in classes}
print('Log priors:', log_priors)

In [ ]:
def predict(text):
    toks = tokenize(text)
    cvec = Counter(toks)
    scores = {}
    for c in classes:
        s = log_priors[c]  # start with log prior
        # add log likelihoods for each word count times
        for w, cnt in cvec.items():
            if w in w2i:
                s += cnt * log_pw_given[c][w2i[w]]
            else:
                # unseen words get uniform probability via smoothing
                s += cnt * np.log(alpha / (total_words[c] + alpha * len(vocab)))
        scores[c] = s
    # return class with highest score
    best = max(scores, key=scores.get)
    return best, scores

def predict_proba(text):
    _, scores = predict(text)
    # convert log-scores to probabilities (unnormalized), then normalize
    vals = np.array([np.exp(scores[c]) for c in classes])
    probs = vals / vals.sum()
    return {c: probs[i] for i,c in enumerate(classes)}

In [ ]:
# Test on a few training and new examples
tests = [texts[0], texts[4], 'free tickets now', 'project due tomorrow']
for t in tests:
    label, scores = predict(t)
    probs = predict_proba(t)
    print(f'
 -> predicted: {label}, probs: {probs}, log-scores: {scores}')

**Interpretation:** The model uses word frequencies and class priors. Words like `free`, `prize`, `tickets` push the log-likelihood toward `spam` because they occur more often in spam examples; words like `meeting`, `project` favor `ham`. Laplace smoothing ensures unseen words don't yield zero probability.